<a href="https://colab.research.google.com/github/MaycatXD/database-smarquest/blob/main/socketio_sqapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install dependencies
!pip install python-socketio eventlet flask pyngrok

import socketio
import eventlet
import eventlet.wsgi
from flask import Flask, jsonify
from pyngrok import ngrok

# --- KONFIGURASI NGROK ---
NGROK_TOKEN = "3B0aoTevfosG4AS4JuvCd25eCYY_zwbuF63s4zonmmmjabaf"
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Setup Socket.IO dan Flask
# async_mode='eventlet' penting untuk kompatibilitas server-client
sio = socketio.Server(cors_allowed_origins='*', async_mode='eventlet')
app = Flask(__name__)
app.wsgi_app = socketio.WSGIApp(sio, app.wsgi_app)

# Route HTTP dengan respon JSON
@app.route('/')
def index():
    return jsonify({
        "status": "running",
        "message": "Socket.IO Server is active",
        "version": "1.0.0",
        "client_compatible": True
    })

# Event Socket.IO
@sio.event
def connect(sid, environ):
    print(f'Client React terhubung: {sid}')
    sio.emit('status', {'msg': 'Connected to backend!'}, room=sid)

# Handler untuk pesan umum dari React
@sio.on('message')
def handle_message(sid, data):
    print(f'Pesan dari {sid}: {data}')
    # Balas ke client atau broadcast
    sio.emit('response', {'data': 'Pesan diterima', 'echo': data})

@sio.event
def disconnect(sid):
    print(f'Client terputus: {sid}')

# 3. Jalankan Tunnel dan Server
PORT = 5000
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)

    public_url = ngrok.connect(PORT).public_url
    print(f"\n[!] URL PUBLIK (Gunakan di React): {public_url}")

    if __name__ == '__main__':
        print("Server mendengarkan di port 5000...")
        eventlet.wsgi.server(eventlet.listen(('', PORT)), app)
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.6/364.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.5 MB/s eta 0:00:00


/tmp/ipykernel_387/1007141482.py:5: DeprecationWarning: 
Eventlet is deprecated. It is currently being maintained in bugfix mode, and
we strongly recommend against using it for new projects.

If you are already using Eventlet, we recommend migrating to a different
framework.  For more detail see
https://eventlet.readthedocs.io/en/latest/asyncio/migration.html

  import eventlet



[!] URL PUBLIK (Gunakan di React): https://tony-jena-regardful.ngrok-free.dev
Server mendengarkan di port 5000...


(387) wsgi starting up on http://0.0.0.0:5000
(387) accepted ('127.0.0.1', 48432)
2402:e100:f64:136a:c8fd:2a12:bb47:8d42,127.0.0.1 - - [17/Mar/2026 04:04:02] "GET / HTTP/1.1" 200 212 0.001904
(387) accepted ('127.0.0.1', 59112)


Client React terhubung: 3FOw8qCSlVENXblmAAAB
Client terputus: 3FOw8qCSlVENXblmAAAB


2402:e100:f64:136a:c8fd:2a12:bb47:8d42,127.0.0.1 - - [17/Mar/2026 04:07:48] "GET /socket.io/?uid=PxUH1QYlxnP2zCvrNa8KtexZXE93&EIO=4&transport=websocket HTTP/1.1" 200 0 171.811003
